# bias_contour_final

Deze notebook werkt met alle `J000`-afbeeldingen uit `/home/yentl/pytorch_gammanet/Images_all_layers_final`.

Opbouw:
1. mean activation heatmaps voor `h0-h4` en `td_h0-td_h3` bij de laatste timestep;
2. temporal dynamics voor `h0-h4`;
3. mean activation heatmaps over timesteps voor `h1`;
4. mask-based quantification in `h1`, inclusief contour enrichment;
5. classificatie van C- en straight-selective channels op basis van `C - straight` contour enrichment.

In [1]:
# ============================================================
# Imports, paths, settings
# ============================================================
import os, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

PROJECT_ROOT = Path("/home/yentl/pytorch_gammanet")
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoint_epoch_40.pt"
INPUT_SIZE = (320, 320)
USE_ABS_ACTIVATION = True
TOP_ACTIVATION_PERCENTILE = 90

BOTTOM_UP_LAYERS = ["h0_exc", "h1_exc", "h2_exc", "h3_exc", "h4_exc"]
TOP_DOWN_LAYERS = ["td_h0_exc", "td_h1_exc", "td_h2_exc", "td_h3_exc"]
ALL_ANALYSIS_LAYERS = BOTTOM_UP_LAYERS + TOP_DOWN_LAYERS
CONTOURS = ["C", "straight"]

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
])

IMAGE_DIR = PROJECT_ROOT / "Images_all_layers_final"
MASK_DIR = PROJECT_ROOT / "contour_masks_all_layers_final"
OUTPUT_DIR = PROJECT_ROOT / "outputs_bias_contour_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_LAYER = "h1_exc"
print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


Using device: cpu
IMAGE_DIR: /home/yentl/pytorch_gammanet/Images_all_layers_final
MASK_DIR: /home/yentl/pytorch_gammanet/contour_masks_all_layers_final
OUTPUT_DIR: /home/yentl/pytorch_gammanet/outputs_bias_contour_final


## Belangrijk over `make_contour_masks.py`

In je externe `make_contour_masks.py` moet je voor deze notebook twee dingen aanpassen als je het script buiten de notebook runt:

```python
IMAGE_DIR = Path("/home/yentl/pytorch_gammanet/Images_all_layers_final")
MASK_DIR = Path("/home/yentl/pytorch_gammanet/contour_masks_all_layers_final")
```

En bij de straight-template:

```python
n_points = 7
```

In deze notebook zit dezelfde mask-code ook ingebouwd, dus je hoeft het externe script niet per se eerst te draaien. De notebook maakt ontbrekende maskers automatisch aan.

In [ ]:
# ============================================================
# Model, parsing, activation and mask helpers
# ============================================================
def load_model():
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    from gammanet.models.vgg16_gammanet_v2 import VGG16GammaNetV2

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    if "config" in checkpoint and "model" in checkpoint["config"]:
        model_config = checkpoint["config"]["model"]
    elif "model_config" in checkpoint:
        model_config = checkpoint["model_config"]
    else:
        raise KeyError("Could not find model config in checkpoint.")

    # Force 4 timesteps if you want to match the trained 4-timestep model behaviour.
    # Leave this commented if the checkpoint config already stores timesteps=4.
    # model_config["timesteps"] = 4

    model = VGG16GammaNetV2(model_config)
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", None))
    if state_dict is None:
        raise KeyError("Could not find model_state_dict or state_dict in checkpoint.")
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    model.to(DEVICE)
    model.eval()
    print("Model timesteps:", model.timesteps)
    return model


def normalize_contour_label(label):
    lower = str(label).strip().lower()
    if lower in ["c", "ccontour", "c_contour"]:
        return "C"
    if lower in ["straight", "line", "straightline", "straight_line"]:
        return "straight"
    return label


def parse_image_filename(path):
    """
    Supports both filename styles:
    - C_BL_0_J000_000.png
    - C_high_BL_0_J000_000.png
    """
    parts = path.stem.split("_")
    if len(parts) == 5:
        contour, quadrant, position, jitter, stim_id = parts
        contrast = "NA"
    elif len(parts) >= 6:
        contour, contrast, quadrant, position, jitter, stim_id = parts[:6]
    else:
        return None
    contour_type = normalize_contour_label(contour)
    if contour_type not in CONTOURS:
        return None
    try:
        return {
            "filename": path.name,
            "path": str(path),
            "contour_type": contour_type,
            "contrast": contrast,
            "quadrant": quadrant,
            "position": int(position),
            "jitter": int(str(jitter).replace("J", "")),
            "stimulus_id": int(stim_id),
        }
    except Exception:
        return None


def load_image_tensor(path):
    pil_img = Image.open(path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
    return pil_img, img_tensor


def reset_and_forward(model, img_tensor):
    model.reset_hidden_states()
    with torch.no_grad():
        return model(img_tensor)


def get_state(model, layer_name):
    state = getattr(model, layer_name, None)
    if state is None:
        raise ValueError(f"{layer_name} is None. Run model first or check layer name.")
    return state


def normalize_for_plot(x, eps=1e-8):
    x = np.asarray(x)
    x = x - np.nanmin(x)
    return x / (np.nanmax(x) + eps)


def resize_map_to_image(fmap, pil_img):
    fmap_t = torch.tensor(fmap, dtype=torch.float32)[None, None]
    resized = F.interpolate(fmap_t, size=pil_img.size[::-1], mode="bilinear", align_corners=False)
    return resized[0, 0].numpy()


def all_channel_map(state, use_abs=True, channels=None):
    x = state.detach()
    if channels is not None:
        channels = [int(c) for c in channels]
        x = x[:, channels, :, :]
    if use_abs:
        x = x.abs()
    return x.mean(dim=1)[0].cpu().numpy()


def single_channel_map(state, channel):
    return state.detach().cpu()[0, int(channel)].numpy()

# ---------- Mask maker: straight now has 7 line elements ----------
N = 512
LINE_WIDTH = 18
STRAIGHT_N_POINTS = 7  # <- pas dit aan als je straight-masker meer/minder streepjes moet volgen
STRAIGHT_EDGE_MARGIN = 0.07


def bezier_curve_position(t, Ps):
    t = np.asarray(t)
    P0, P1, P2, P3 = Ps
    return (((1 - t) ** 3)[:, None] * P0 +
            (3 * ((1 - t) ** 2) * t)[:, None] * P1 +
            (3 * (1 - t) * (t ** 2))[:, None] * P2 +
            (t ** 3)[:, None] * P3)


def build_templates():
    Ps = np.array([[0.75, 0.9], [0.2, 0.9], [0.2, 0.1], [0.75, 0.1]])
    curve = bezier_curve_position(np.linspace(0, 1, 200), Ps * 0.5)
    bx_base = curve[:, 0][20:-20]
    by_base = curve[:, 1][20:-20]
    bx_base = bx_base - np.min(bx_base) + 0.07
    by_base = by_base - np.max(by_base) + 0.94
    templates = {}
    for shift_idx in range(4):
        bx = bx_base + 0.1 * shift_idx
        by = by_base.copy()
        templates[("C", shift_idx)] = (bx, by)
        x_center = np.mean(bx)
        templates[("bC", shift_idx)] = (2 * x_center - bx, by)
        bx_straight = np.full(STRAIGHT_N_POINTS, np.mean(bx))
        by_straight = np.linspace(np.min(by) + STRAIGHT_EDGE_MARGIN, np.max(by) - STRAIGHT_EDGE_MARGIN, STRAIGHT_N_POINTS)
        templates[("straight", shift_idx)] = (bx_straight, by_straight)
    return templates

TEMPLATES = build_templates()


def draw_template_mask(shape, position):
    bx, by = TEMPLATES[(shape, position)]
    points = list(zip(bx * N, by * N))
    mask = Image.new("L", (N, N), 0)
    draw = ImageDraw.Draw(mask)
    if shape == "straight":
        draw.line(points, fill=255, width=LINE_WIDTH)
    else:
        draw.line(points, fill=255, width=LINE_WIDTH, joint="curve")
    return mask


def apply_quadrant_transform(mask, original_shape, quadrant):
    effective_shape = original_shape
    if quadrant == "BL":
        pass
    elif quadrant == "BR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if original_shape == "C": effective_shape = "bC"
        elif original_shape == "bC": effective_shape = "C"
    elif quadrant == "TL":
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
    elif quadrant == "TR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM)
        if original_shape == "C": effective_shape = "bC"
        elif original_shape == "bC": effective_shape = "C"
    else:
        raise ValueError(f"Unknown quadrant: {quadrant}")
    return mask, effective_shape


def make_mask_for_row(row, mask_dir):
    wanted_shape = row["contour_type"]
    quadrant = row["quadrant"]
    position = int(row["position"])
    base_shapes = ["straight"] if wanted_shape == "straight" else ["C", "bC"]
    candidate_masks = []
    for base_shape in base_shapes:
        mask = draw_template_mask(base_shape, position)
        mask, effective_shape = apply_quadrant_transform(mask, base_shape, quadrant)
        if effective_shape == wanted_shape:
            candidate_masks.append(mask)
    if not candidate_masks:
        return False
    final = Image.new("L", (N, N), 0)
    for mask in candidate_masks:
        final = Image.fromarray(np.maximum(np.asarray(final), np.asarray(mask)).astype(np.uint8))
    save_path = Path(mask_dir) / row["filename"]
    final.save(save_path)
    return True


def ensure_masks(df, mask_dir):
    mask_dir = Path(mask_dir)
    mask_dir.mkdir(parents=True, exist_ok=True)
    n_made, n_exists = 0, 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking/making masks"):
        if (mask_dir / row["filename"]).exists():
            n_exists += 1
        else:
            n_made += int(make_mask_for_row(row, mask_dir))
    print(f"Masks already present: {n_exists}; masks created: {n_made}; mask dir: {mask_dir}")


def load_true_contour_mask(row, target_size, mask_dir):
    mask_path = Path(mask_dir) / row["filename"]
    if not mask_path.exists():
        make_mask_for_row(row, mask_dir)
    mask = Image.open(mask_path).convert("L")
    mask = mask.resize(target_size, resample=Image.NEAREST)
    return np.asarray(mask) > 0


def compute_mask_metrics_from_activation(act, contour_mask):
    act = np.asarray(act)
    act = np.abs(act) if USE_ABS_ACTIVATION else act.copy()
    act = np.nan_to_num(act, nan=0.0, posinf=0.0, neginf=0.0)
    act = act - act.min() + 1e-8
    background_mask = ~contour_mask
    contour_values = act[contour_mask]
    background_values = act[background_mask]
    contour_mean = contour_values.mean()
    background_mean = background_values.mean()
    contour_sum = contour_values.sum()
    total_sum = act.sum()
    contour_area_pct = 100 * contour_mask.mean()
    activation_on_contour_pct = 100 * contour_sum / (total_sum + 1e-8)
    contour_enrichment = activation_on_contour_pct / (contour_area_pct + 1e-8)
    act_threshold = np.percentile(act, TOP_ACTIVATION_PERCENTILE)
    top_activation_mask = act >= act_threshold
    intersection = np.logical_and(contour_mask, top_activation_mask).sum()
    dice_top_activation = 2 * intersection / (contour_mask.sum() + top_activation_mask.sum() + 1e-8)
    return {
        "contour_mean_activation": contour_mean,
        "background_mean_activation": background_mean,
        "contour_preference_ratio": contour_mean / (background_mean + 1e-8),
        "activation_on_contour_pct": activation_on_contour_pct,
        "contour_area_pct": contour_area_pct,
        "contour_enrichment": contour_enrichment,
        "dice_top_activation": dice_top_activation,
        "top_activation_percentile": TOP_ACTIVATION_PERCENTILE,
    }


def plot_state_overlay(state, pil_img, title, save_path=None, channels=None, alpha=0.55):
    fmap = all_channel_map(state, use_abs=USE_ABS_ACTIVATION, channels=channels)
    fmap_resized = resize_map_to_image(fmap, pil_img)
    fmap_norm = normalize_for_plot(fmap_resized)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(pil_img); axes[0].set_title("Stimulus"); axes[0].axis("off")
    im = axes[1].imshow(fmap_norm, cmap="inferno"); axes[1].set_title(title); axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    axes[2].imshow(pil_img); axes[2].imshow(fmap_norm, cmap="inferno", alpha=alpha)
    axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path); save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()
    plt.close(fig)


In [ ]:
# ============================================================
# Load model and image table
# ============================================================
model = load_model()

rows = []
for path in sorted(IMAGE_DIR.glob("*.png")):
    info = parse_image_filename(path)
    if info is not None and info["jitter"] == 0:
        rows.append(info)

image_df = pd.DataFrame(rows)
if image_df.empty:
    raise RuntimeError(f"No J000 C/straight images found in {IMAGE_DIR}")

image_df = image_df.sort_values(["contour_type", "position", "quadrant", "stimulus_id"]).reset_index(drop=True)
print(image_df.groupby("contour_type").size())
display(image_df.head())

(OUTPUT_DIR / "csv").mkdir(parents=True, exist_ok=True)
image_df.to_csv(OUTPUT_DIR / "csv" / "image_table_J000.csv", index=False)


In [ ]:
# ============================================================
# Collect mean final states and temporal bottom-up states
# ============================================================
final_sum = {c: {layer: None for layer in ALL_ANALYSIS_LAYERS} for c in CONTOURS}
final_count = {c: 0 for c in CONTOURS}
temporal_sum = {c: {layer: None for layer in BOTTOM_UP_LAYERS} for c in CONTOURS}
temporal_count = {c: 0 for c in CONTOURS}

for _, row in tqdm(image_df.iterrows(), total=len(image_df), desc="Forward J000 images"):
    contour = row["contour_type"]
    pil_img, img_tensor = load_image_tensor(row["path"])
    reset_and_forward(model, img_tensor)

    for layer in ALL_ANALYSIS_LAYERS:
        state = get_state(model, layer).detach().cpu()
        final_sum[contour][layer] = state.clone() if final_sum[contour][layer] is None else final_sum[contour][layer] + state
    final_count[contour] += 1

    if hasattr(model, "temporal_activity"):
        for layer in BOTTOM_UP_LAYERS:
            if layer in model.temporal_activity:
                stack = torch.stack(model.temporal_activity[layer], dim=0)  # [T, B, C, H, W]
                temporal_sum[contour][layer] = stack.clone() if temporal_sum[contour][layer] is None else temporal_sum[contour][layer] + stack
        temporal_count[contour] += 1

mean_states = {c: {} for c in CONTOURS}
temporal_mean_states = {c: {} for c in CONTOURS}
for c in CONTOURS:
    for layer in ALL_ANALYSIS_LAYERS:
        mean_states[c][layer] = final_sum[c][layer] / final_count[c]
    for layer in BOTTOM_UP_LAYERS:
        if temporal_sum[c][layer] is not None:
            temporal_mean_states[c][layer] = temporal_sum[c][layer] / temporal_count[c]

print("Done. Counts:", final_count)


In [ ]:
# ============================================================
# 1. Mean activation heatmaps: final timestep, all layers
# ============================================================
MEAN_HEATMAP_DIR = OUTPUT_DIR / "plots" / "01_mean_heatmaps_final_timestep"
MEAN_HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

for contour in CONTOURS:
    example_row = image_df[image_df["contour_type"] == contour].iloc[0]
    pil_img, _ = load_image_tensor(example_row["path"])
    for layer in ALL_ANALYSIS_LAYERS:
        plot_state_overlay(
            mean_states[contour][layer],
            pil_img,
            title=f"{contour} | {layer} | mean final activation",
            save_path=MEAN_HEATMAP_DIR / f"mean_final_{contour}_{layer}.png",
        )


In [ ]:
# ============================================================
# 2. Temporal dynamics: h0-h4
# ============================================================
records = []
for contour in CONTOURS:
    for layer in BOTTOM_UP_LAYERS:
        if layer not in temporal_mean_states[contour]:
            continue
        stack = temporal_mean_states[contour][layer]
        for t in range(stack.shape[0]):
            records.append({
                "contour_type": contour,
                "layer": layer,
                "timestep": t,
                "mean_abs_activation": stack[t].abs().mean().item(),
                "n_images": temporal_count[contour],
            })

temporal_df = pd.DataFrame(records)
temporal_df.to_csv(OUTPUT_DIR / "csv" / "temporal_dynamics_h0_h4.csv", index=False)
display(temporal_df.head())

TEMPORAL_DIR = OUTPUT_DIR / "plots" / "02_temporal_dynamics"
TEMPORAL_DIR.mkdir(parents=True, exist_ok=True)

for layer in BOTTOM_UP_LAYERS:
    sub = temporal_df[temporal_df["layer"] == layer]
    fig, ax = plt.subplots(figsize=(6, 4))
    for contour in CONTOURS:
        csub = sub[sub["contour_type"] == contour]
        ax.plot(csub["timestep"], csub["mean_abs_activation"], marker="o", label=contour)
    ax.set_title(f"Temporal dynamics | {layer}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Mean absolute activation")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_path = TEMPORAL_DIR / f"temporal_{layer}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show(); plt.close(fig)


In [ ]:
# ============================================================
# 3. Mean activation heatmaps over timesteps: h1 only
# ============================================================
TIMESTEP_DIR = OUTPUT_DIR / "plots" / "03_h1_heatmaps_per_timestep"
TIMESTEP_DIR.mkdir(parents=True, exist_ok=True)

for contour in CONTOURS:
    example_row = image_df[image_df["contour_type"] == contour].iloc[0]
    pil_img, _ = load_image_tensor(example_row["path"])
    stack = temporal_mean_states[contour][ANALYSIS_LAYER]
    for t in range(stack.shape[0]):
        plot_state_overlay(
            stack[t],
            pil_img,
            title=f"{contour} | {ANALYSIS_LAYER} | timestep {t}",
            save_path=TIMESTEP_DIR / f"{contour}_{ANALYSIS_LAYER}_timestep_{t}.png",
        )


In [ ]:
# ============================================================
# 4. Create/check masks and quantify h1 per channel
# ============================================================
ensure_masks(image_df, MASK_DIR)

# Visual sanity check
MASK_CHECK_DIR = OUTPUT_DIR / "plots" / "04_mask_checks"
MASK_CHECK_DIR.mkdir(parents=True, exist_ok=True)
for contour in CONTOURS:
    row = image_df[image_df["contour_type"] == contour].iloc[0]
    pil_img, _ = load_image_tensor(row["path"])
    mask = load_true_contour_mask(row, target_size=pil_img.size, mask_dir=MASK_DIR)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(pil_img)
    ax.imshow(mask, alpha=0.35)
    ax.set_title(f"Mask sanity check | {contour}")
    ax.axis("off")
    save_path = MASK_CHECK_DIR / f"mask_check_{contour}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show(); plt.close(fig)

metric_rows = []
for _, row in tqdm(image_df.iterrows(), total=len(image_df), desc="Quantify h1 channels"):
    pil_img, img_tensor = load_image_tensor(row["path"])
    reset_and_forward(model, img_tensor)
    state = get_state(model, ANALYSIS_LAYER).detach().cpu()
    contour_mask = load_true_contour_mask(row, target_size=pil_img.size, mask_dir=MASK_DIR)
    for channel in range(state.shape[1]):
        fmap = single_channel_map(state, channel)
        fmap_resized = resize_map_to_image(fmap, pil_img)
        metrics = compute_mask_metrics_from_activation(fmap_resized, contour_mask)
        metric_rows.append({
            **{k: row[k] for k in ["filename", "contour_type", "contrast", "quadrant", "position", "jitter", "stimulus_id"]},
            "layer": ANALYSIS_LAYER,
            "channel": int(channel),
            **metrics,
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(OUTPUT_DIR / "csv" / "h1_channel_mask_metrics_per_image.csv", index=False)

mean_channel_df = (
    metrics_df
    .groupby(["contour_type", "layer", "channel"], as_index=False)
    .agg(
        n_images=("filename", "nunique"),
        mean_contour_mean_activation=("contour_mean_activation", "mean"),
        mean_background_mean_activation=("background_mean_activation", "mean"),
        mean_contour_preference_ratio=("contour_preference_ratio", "mean"),
        mean_activation_on_contour_pct=("activation_on_contour_pct", "mean"),
        mean_contour_area_pct=("contour_area_pct", "mean"),
        mean_contour_enrichment=("contour_enrichment", "mean"),
        mean_dice_top_activation=("dice_top_activation", "mean"),
    )
)
mean_channel_df.to_csv(OUTPUT_DIR / "csv" / "h1_channel_mask_metrics_mean_by_contour.csv", index=False)
display(mean_channel_df.head())


In [ ]:
# ============================================================
# 5. Classification of contour-selective channels
#    C-minus-straight contour enrichment
# ============================================================
CLASSIFICATION_DIR = OUTPUT_DIR / "csv" / "channel_classification"
CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)

wide = mean_channel_df.pivot(index="channel", columns="contour_type", values="mean_contour_enrichment").reset_index()
wide["C_minus_straight_enrichment"] = wide["C"] - wide["straight"]
wide["channel_class"] = np.where(wide["C_minus_straight_enrichment"] > 0, "C_channel", "straight_channel")
wide = wide.sort_values("C_minus_straight_enrichment", ascending=False)

C_CHANNELS = wide.loc[wide["C_minus_straight_enrichment"] > 0, "channel"].astype(int).tolist()
STRAIGHT_CHANNELS = wide.loc[wide["C_minus_straight_enrichment"] < 0, "channel"].astype(int).tolist()

wide.to_csv(CLASSIFICATION_DIR / "h1_channel_classes_from_mean_enrichment.csv", index=False)
pd.DataFrame({"channel": C_CHANNELS}).to_csv(CLASSIFICATION_DIR / "C_channels_h1.csv", index=False)
pd.DataFrame({"channel": STRAIGHT_CHANNELS}).to_csv(CLASSIFICATION_DIR / "straight_channels_h1.csv", index=False)

print(f"C channels: {len(C_CHANNELS)}")
print(f"Straight channels: {len(STRAIGHT_CHANNELS)}")
display(wide.head(20))
display(wide.tail(20))

# Small overview plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(len(wide)), wide["C_minus_straight_enrichment"].values)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Channels sorted by C - straight enrichment")
ax.set_ylabel("C - straight contour enrichment")
ax.set_title("h1 channel classification")
plt.tight_layout()
save_path = OUTPUT_DIR / "plots" / "05_channel_classification" / "h1_C_minus_straight_enrichment.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved:", save_path)
plt.show(); plt.close(fig)
